# Capstone Project One: Diabetes Hospital Readmission Analytics

**Role:** Data Analyst for a hospital system  
**Framework:** CRISP-DM  
**Tools:** Python, Pandas, Matplotlib, SQL/SQLite, Power BI

## Main Business Question
**What patient, hospitalization, utilization, diagnosis, and medication characteristics are associated with hospital readmission?**

> This is a descriptive analytics project. The analysis identifies patterns and associations; it does not establish causation.

## 1. Business Understanding

The analysis answers the following business questions:

1. What is the overall hospital readmission rate?
2. How does readmission vary by gender?
3. Which age groups have higher readmission rates?
4. How does readmission vary by race?
5. Which admission types have higher readmission rates?
6. Which discharge dispositions have higher readmission rates?
7. Does length of hospital stay differ between readmitted and non-readmitted encounters?
8. How do previous inpatient visits relate to readmission?
9. How do previous emergency visits relate to readmission?
10. How do previous outpatient visits relate to readmission?
11. How does overall prior healthcare utilization relate to readmission?
12. Which primary diagnoses have high readmission rates while also representing meaningful encounter volume?
13. How does diabetes medication status relate to readmission?
14. How does a diabetes medication change relate to readmission?
15. Which age and admission-type segments have higher readmission rates?
16. Are encounters with longer-than-average stays for their age group more likely to be readmitted?

### Required KPIs
Total Encounters, Total Readmissions, Readmission Rate, Average Length of Stay, Average Medications, Average Lab Procedures, Average Prior Inpatient Visits, Average Emergency Visits, Percentage Receiving Diabetes Medication, and Percentage with Medication Changes.

## 2. Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Load the raw dataset.
# Keep the CSV in the same folder as this notebook, or update the path below.
df = pd.read_csv("diabete_huggingface.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
# Review columns and data types.
df.info()

In [ ]:
# Numeric descriptive statistics.
df.describe()

In [ ]:
# Categorical descriptive statistics.
df.describe(include="object")

In [ ]:
# Inspect important categorical variables.
for col in ["readmitted", "gender", "race", "age"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

### Data Quality Profile

In [ ]:
# Create a compact data-quality profile.
profile = pd.DataFrame({
    "Data_Type": df.dtypes,
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": df.isna().mean() * 100,
    "Unique_Values": df.nunique(dropna=True)
}).sort_values("Missing_Percent", ascending=False)

profile.head(20)

In [ ]:
# The raw data also uses '?' as a special missing-value indicator.
question_marks = (
    df.astype(str)
      .eq("?")
      .sum()
      .sort_values(ascending=False)
)

question_marks[question_marks > 0]

In [ ]:
# Duplicate and identifier checks.
print("Duplicate rows:", df.duplicated().sum())
print("Unique row IDs:", df["rowID"].nunique())
print("Total rows:", len(df))

### Data Quality Interpretation
After replacing the special `?` indicator with missing values, the most important missing-data issues should be reviewed before interpretation. In the completed analysis, `max_glu_serum` and `A1Cresult` contain very high missingness, so conclusions based on those fields should be cautious. Small categories should also be interpreted with encounter volume in mind.

## 3. Data Preparation and Feature Engineering

In [ ]:
# IMPORTANT: use .copy() so the raw dataframe remains unchanged.
df_clean = df.copy()

# Replace the special missing-value indicator with NaN.
df_clean.replace("?", np.nan, inplace=True)

# Numeric binary readmission indicator: 1 = readmitted, 0 = not readmitted.
df_clean["readmission_flag"] = df_clean["readmitted"].astype(int)

# Total prior healthcare utilization.
df_clean["prior_utilization"] = (
    df_clean["number_inpatient"]
    + df_clean["number_outpatient"]
    + df_clean["number_emergency"]
)

# Utilization categories required for the project.
df_clean["utilization_group"] = pd.cut(
    df_clean["prior_utilization"],
    bins=[-1, 0, 2, 5, float("inf")],
    labels=["None", "Low", "Medium", "High"]
)

# Length-of-stay categories.
df_clean["stay_group"] = pd.cut(
    df_clean["time_in_hospital"],
    bins=[0, 3, 7, 14],
    labels=["Short", "Medium", "Long"]
)

df_clean.head()

In [ ]:
# Missingness after cleaning.
missing_summary = (
    df_clean.isna()
            .mean()
            .mul(100)
            .sort_values(ascending=False)
)

missing_summary.head(15)

## 4. KPI Summary

In [ ]:
kpis = {
    "Total Encounters": len(df_clean),
    "Total Readmissions": int(df_clean["readmission_flag"].sum()),
    "Readmission Rate (%)": df_clean["readmission_flag"].mean() * 100,
    "Average Length of Stay": df_clean["time_in_hospital"].mean(),
    "Average Medications": df_clean["num_medications"].mean(),
    "Average Lab Procedures": df_clean["num_lab_procedures"].mean(),
    "Average Prior Inpatient Visits": df_clean["number_inpatient"].mean(),
    "Average Emergency Visits": df_clean["number_emergency"].mean(),
    "Receiving Diabetes Medication (%)": (df_clean["diabetesMed"] == "Yes").mean() * 100,
    "Medication Change (%)": (df_clean["change"] == "Ch").mean() * 100
}

pd.Series(kpis).round(2).to_frame("Value")

**Verified results from the completed analysis:** 10,000 encounters; 3,965 readmissions; 39.65% overall readmission rate; average LOS 4.43 days; average medications 15.56; average lab procedures 43.08; average prior inpatient visits 0.39; average emergency visits 0.12; 74.78% receiving diabetes medication; 42.76% with medication changes.

## 5. Advanced Pandas Analysis

### Q1–Q4. Overall Rate and Patient Characteristics

In [ ]:
# Gender summary: groupby + agg + sort_values.
gender_summary = (
    df_clean.groupby("gender")
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

gender_summary["Readmission_Rate"] *= 100
gender_summary.sort_values("Readmission_Rate", ascending=False)

In [ ]:
# Age summary.
age_summary = (
    df_clean.groupby("age", observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

age_summary["Readmission_Rate"] *= 100
age_summary.sort_values("Readmission_Rate", ascending=False)

In [ ]:
# Race summary.
race_summary = (
    df_clean.groupby("race", observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

race_summary["Readmission_Rate"] *= 100
race_summary.sort_values("Readmission_Rate", ascending=False)

**Interpretation:** Female encounters had a slightly higher readmission rate than male encounters (40.22% vs. 38.98%). Readmission generally increased across older age groups, with the 80–90 group at 44.32% and the 70–80 group at 43.04%. Because the 70–80 group also contains 2,595 encounters, it is operationally meaningful.

### Q5–Q7. Hospitalization Characteristics

In [ ]:
# Admission type.
admission_summary = (
    df_clean.groupby("admission_type_id", observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

admission_summary["Readmission_Rate"] *= 100
admission_summary.sort_values("Readmission_Rate", ascending=False)

In [ ]:
# Discharge disposition.
discharge_summary = (
    df_clean.groupby("discharge_disposition_id", observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

discharge_summary["Readmission_Rate"] *= 100
discharge_summary.sort_values("Readmission_Rate", ascending=False)

In [ ]:
# Length of stay by readmission status.
los_summary = (
    df_clean.groupby("readmission_flag")
    .agg(
        Encounters=("rowID", "count"),
        Average_LOS=("time_in_hospital", "mean")
    )
    .reset_index()
)

los_summary["Average_LOS"] = los_summary["Average_LOS"].round(2)
los_summary

**Interpretation:** Emergency admissions represented a large group and had a 40.12% readmission rate. Readmitted encounters had a slightly longer average LOS (4.63 days) than non-readmitted encounters (4.31 days). These are associations and should not be interpreted as causal effects.

### Q8–Q11. Prior Healthcare Utilization

In [ ]:
def utilization_summary(column):
    summary = (
        df_clean.groupby(column)
        .agg(
            Encounters=("rowID", "count"),
            Readmissions=("readmission_flag", "sum"),
            Readmission_Rate=("readmission_flag", "mean")
        )
        .reset_index()
    )
    summary["Readmission_Rate"] *= 100
    return summary

inpatient_summary = utilization_summary("number_inpatient")
emergency_summary = utilization_summary("number_emergency")
outpatient_summary = utilization_summary("number_outpatient")

inpatient_summary.head(10)

In [ ]:
emergency_summary.head(10)

In [ ]:
outpatient_summary.head(10)

In [ ]:
# Overall utilization groups created with pd.cut().
overall_utilization_summary = (
    df_clean.groupby("utilization_group", observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

overall_utilization_summary["Readmission_Rate"] *= 100
overall_utilization_summary

**Interpretation:** Prior inpatient use shows one of the clearest patterns. Encounters with no prior inpatient visits had a 34.36% readmission rate, compared with 51.17% for one prior visit and 62.74% for two. Prior emergency use shows a similar pattern. Very high percentages at extreme visit counts are based on very small groups and should be interpreted cautiously.

### Q12. Diagnosis Analysis: Rate + Volume

In [ ]:
diagnosis_summary = (
    df_clean.groupby("diag_1_desc")
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

diagnosis_summary["Readmission_Rate"] *= 100

# query() removes very small diagnosis groups.
diagnosis_summary = diagnosis_summary.query("Encounters >= 30").copy()

# rank() demonstrates diagnosis risk ranking.
diagnosis_summary["Risk_Rank"] = (
    diagnosis_summary["Readmission_Rate"]
    .rank(method="dense", ascending=False)
)

diagnosis_summary.sort_values(
    ["Readmission_Rate", "Encounters"],
    ascending=[False, False]
).head(15)

**Interpretation:** The diagnosis with the highest rate is not necessarily the diagnosis with the greatest operational impact. Congestive heart failure is especially important because it combines a high readmission rate (53.77%) with substantial volume: 636 encounters and 342 readmissions.

### Q13–Q14. Diabetes Medication and Medication Change

In [ ]:
medication_summary = (
    df_clean.groupby("diabetesMed")
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)
medication_summary["Readmission_Rate"] *= 100
medication_summary

In [ ]:
change_summary = (
    df_clean.groupby("change")
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)
change_summary["Readmission_Rate"] *= 100
change_summary

**Interpretation:** Encounters receiving diabetes medication had a 41.16% readmission rate versus 35.17% without diabetes medication. Encounters with a medication change had a 42.54% rate versus 37.49% without a change. These differences may reflect underlying clinical complexity; they do not show that medication treatment causes readmission.

### Q15. Multi-Column Segment Analysis

In [ ]:
# Multi-column grouping.
age_admission_summary = (
    df_clean.groupby(["age", "admission_type_id"], observed=True)
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean")
    )
    .reset_index()
)

age_admission_summary["Readmission_Rate"] *= 100

# query() keeps segments large enough to interpret more reliably.
age_admission_summary = (
    age_admission_summary
    .query("Encounters >= 30")
    .sort_values("Readmission_Rate", ascending=False)
)

age_admission_summary.head(15)

### Q16. transform() and .loc[]: LOS Relative to Age-Group Average

In [ ]:
# transform() calculates the age-group mean while preserving row-level data.
df_clean["age_avg_los"] = (
    df_clean.groupby("age", observed=True)["time_in_hospital"]
    .transform("mean")
)

df_clean["los_difference"] = (
    df_clean["time_in_hospital"] - df_clean["age_avg_los"]
)

df_clean["los_vs_age_average"] = np.where(
    df_clean["los_difference"] > 0,
    "Above Age-Group Average",
    "At or Below Age-Group Average"
)

los_age_summary = (
    df_clean.groupby("los_vs_age_average")
    .agg(
        Encounters=("rowID", "count"),
        Readmissions=("readmission_flag", "sum"),
        Readmission_Rate=("readmission_flag", "mean"),
        Average_LOS=("time_in_hospital", "mean")
    )
    .reset_index()
)

los_age_summary["Readmission_Rate"] *= 100
los_age_summary

In [ ]:
# .loc[] example: inspect readmitted encounters with a stay of at least 7 days.
long_readmitted = df_clean.loc[
    (df_clean["readmission_flag"] == 1)
    & (df_clean["time_in_hospital"] >= 7),
    [
        "age",
        "gender",
        "time_in_hospital",
        "number_inpatient",
        "number_emergency",
        "diag_1_desc"
    ]
]

long_readmitted.head()

**Interpretation:** Encounters above their age-group average LOS had a 41.89% readmission rate versus 38.09% for encounters at or below the age-group average.

### Pivot Table and value_counts() Demonstrations

In [ ]:
# pivot_table(): age-by-gender readmission rate.
age_gender_pivot = pd.pivot_table(
    df_clean,
    values="readmission_flag",
    index="age",
    columns="gender",
    aggfunc="mean",
    observed=True
) * 100

age_gender_pivot.round(2)

In [ ]:
# value_counts(): distribution of utilization groups.
df_clean["utilization_group"].value_counts(sort=False)

## 6. Python Data Visualizations

In [ ]:
# Visualization 1: Readmission rate by age.
age_plot = age_summary.sort_values("age")
plt.figure(figsize=(10, 5))
plt.bar(age_plot["age"].astype(str), age_plot["Readmission_Rate"])
plt.title("Readmission Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Length of stay by readmission status.
df_clean.boxplot(column="time_in_hospital", by="readmission_flag")
plt.title("Length of Stay by Readmission Status")
plt.suptitle("")
plt.xlabel("Readmission Flag")
plt.ylabel("Days")
plt.show()

In [ ]:
# Visualization 3: Stacked bar chart by gender.
gender_readmission = pd.crosstab(
    df_clean["gender"],
    df_clean["readmission_flag"],
    normalize="index"
) * 100

gender_readmission.plot(kind="bar", stacked=True, figsize=(7, 5))
plt.title("Readmission Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Percent of Encounters")
plt.legend(["Not Readmitted", "Readmitted"])
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 4: Hospital LOS distribution.
plt.figure(figsize=(8, 5))
plt.hist(df_clean["time_in_hospital"], bins=14)
plt.title("Distribution of Hospital Length of Stay")
plt.xlabel("Days")
plt.ylabel("Encounter Count")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 5: Prior inpatient visits vs. LOS.
plt.figure(figsize=(8, 5))
plt.scatter(df_clean["number_inpatient"], df_clean["time_in_hospital"], alpha=0.25)
plt.title("Prior Inpatient Visits vs. Length of Stay")
plt.xlabel("Prior Inpatient Visits")
plt.ylabel("Length of Stay (Days)")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 6: Age-gender heatmap using Matplotlib.
matrix = age_gender_pivot.to_numpy()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, aspect="auto")

ax.set_xticks(range(len(age_gender_pivot.columns)))
ax.set_xticklabels(age_gender_pivot.columns)
ax.set_yticks(range(len(age_gender_pivot.index)))
ax.set_yticklabels(age_gender_pivot.index)
ax.set_title("Readmission Rate by Age and Gender")
ax.set_xlabel("Gender")
ax.set_ylabel("Age Group")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix[i, j]:.1f}%", ha="center", va="center")

fig.colorbar(im, ax=ax, label="Readmission Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 7: Top diagnoses by readmission rate with meaningful volume.
top_diagnoses = (
    diagnosis_summary
    .sort_values("Readmission_Rate", ascending=False)
    .head(10)
    .sort_values("Readmission_Rate")
)

plt.figure(figsize=(10, 6))
plt.barh(top_diagnoses["diag_1_desc"], top_diagnoses["Readmission_Rate"])
plt.title("Top Primary Diagnoses by Readmission Rate (30+ Encounters)")
plt.xlabel("Readmission Rate (%)")
plt.ylabel("Primary Diagnosis")
plt.tight_layout()
plt.show()

## 7. Export the Clean Dataset

In [ ]:
df_clean.to_csv("diabetes_clean.csv", index=False)
print("Saved: diabetes_clean.csv")

## 8. SQL Analysis

In [ ]:
conn = sqlite3.connect("diabetes_analytics.db")

df_clean.to_sql(
    "hospital_encounters",
    conn,
    if_exists="replace",
    index=False
)

print("Loaded cleaned data into SQLite table: hospital_encounters")

### Query 1 — Overall Readmission KPI

In [ ]:
query = """SELECT
    COUNT(*) AS total_encounters,
    SUM(readmission_flag) AS total_readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters;"""
pd.read_sql_query(query, conn)

### Query 2 — Readmission by Gender

In [ ]:
query = """SELECT
    gender,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY gender
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 3 — Readmission by Age

In [ ]:
query = """SELECT
    age,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY age
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 4 — Readmission by Race

In [ ]:
query = """SELECT
    race,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
WHERE race IS NOT NULL
GROUP BY race
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 5 — Admission Type

In [ ]:
query = """SELECT
    admission_type_id,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
WHERE admission_type_id IS NOT NULL
GROUP BY admission_type_id
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 6 — LOS by Readmission Status

In [ ]:
query = """SELECT
    readmission_flag,
    COUNT(*) AS encounters,
    ROUND(AVG(time_in_hospital), 2) AS average_los
FROM hospital_encounters
GROUP BY readmission_flag;"""
pd.read_sql_query(query, conn)

### Query 7 — Prior Inpatient Visits

In [ ]:
query = """SELECT
    number_inpatient,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY number_inpatient
ORDER BY number_inpatient;"""
pd.read_sql_query(query, conn)

### Query 8 — CASE WHEN Utilization Groups

In [ ]:
query = """SELECT
    CASE
        WHEN prior_utilization = 0 THEN 'None'
        WHEN prior_utilization BETWEEN 1 AND 2 THEN 'Low'
        WHEN prior_utilization BETWEEN 3 AND 5 THEN 'Medium'
        ELSE 'High'
    END AS utilization_group_sql,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY utilization_group_sql
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 9 — Diagnosis with HAVING

In [ ]:
query = """SELECT
    diag_1_desc,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
WHERE diag_1_desc IS NOT NULL
GROUP BY diag_1_desc
HAVING COUNT(*) >= 30
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 10 — Diagnosis CTE

In [ ]:
query = """WITH diagnosis_summary AS (
    SELECT
        diag_1_desc,
        COUNT(*) AS encounters,
        SUM(readmission_flag) AS readmissions,
        100.0 * AVG(readmission_flag) AS readmission_rate
    FROM hospital_encounters
    WHERE diag_1_desc IS NOT NULL
    GROUP BY diag_1_desc
)
SELECT
    diag_1_desc,
    encounters,
    readmissions,
    ROUND(readmission_rate, 2) AS readmission_rate
FROM diagnosis_summary
WHERE encounters >= 30
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 11 — Window Function and Ranking

In [ ]:
query = """WITH diagnosis_summary AS (
    SELECT
        diag_1_desc,
        COUNT(*) AS encounters,
        100.0 * AVG(readmission_flag) AS readmission_rate
    FROM hospital_encounters
    WHERE diag_1_desc IS NOT NULL
    GROUP BY diag_1_desc
    HAVING COUNT(*) >= 30
)
SELECT
    diag_1_desc,
    encounters,
    ROUND(readmission_rate, 2) AS readmission_rate,
    DENSE_RANK() OVER (ORDER BY readmission_rate DESC) AS risk_rank
FROM diagnosis_summary
ORDER BY risk_rank, encounters DESC;"""
pd.read_sql_query(query, conn)

### Query 12 — Conditional Aggregation

In [ ]:
query = """SELECT
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN readmission_flag = 1 THEN 1 ELSE 0 END) AS readmissions,
    SUM(CASE WHEN readmission_flag = 0 THEN 1 ELSE 0 END) AS not_readmitted,
    ROUND(
        100.0 * SUM(CASE WHEN readmission_flag = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS readmission_rate
FROM hospital_encounters;"""
pd.read_sql_query(query, conn)

### Query 13 — Diabetes Medication

In [ ]:
query = """SELECT
    diabetesMed,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY diabetesMed
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

### Query 14 — Medication Change

In [ ]:
query = """SELECT
    change,
    COUNT(*) AS encounters,
    SUM(readmission_flag) AS readmissions,
    ROUND(100.0 * AVG(readmission_flag), 2) AS readmission_rate
FROM hospital_encounters
GROUP BY change
ORDER BY readmission_rate DESC;"""
pd.read_sql_query(query, conn)

## 9. Pandas vs. SQL Validation

In [ ]:
# Independently calculate five KPIs in Pandas.
pandas_validation = {
    "Total Encounters": len(df_clean),
    "Total Readmissions": df_clean["readmission_flag"].sum(),
    "Readmission Rate": df_clean["readmission_flag"].mean() * 100,
    "Average Length of Stay": df_clean["time_in_hospital"].mean(),
    "Average Medications": df_clean["num_medications"].mean()
}

# Independently calculate the same KPIs in SQL.
sql_validation = pd.read_sql_query(
    '''
    SELECT
        COUNT(*) AS total_encounters,
        SUM(readmission_flag) AS total_readmissions,
        100.0 * AVG(readmission_flag) AS readmission_rate,
        AVG(time_in_hospital) AS average_los,
        AVG(num_medications) AS average_medications
    FROM hospital_encounters;
    ''',
    conn
).iloc[0]

validation = pd.DataFrame({
    "KPI": [
        "Total Encounters",
        "Total Readmissions",
        "Readmission Rate",
        "Average Length of Stay",
        "Average Medications"
    ],
    "Pandas Result": [
        pandas_validation["Total Encounters"],
        pandas_validation["Total Readmissions"],
        pandas_validation["Readmission Rate"],
        pandas_validation["Average Length of Stay"],
        pandas_validation["Average Medications"]
    ],
    "SQL Result": [
        sql_validation["total_encounters"],
        sql_validation["total_readmissions"],
        sql_validation["readmission_rate"],
        sql_validation["average_los"],
        sql_validation["average_medications"]
    ]
})

validation["Match"] = np.isclose(
    validation["Pandas Result"],
    validation["SQL Result"]
)

validation.round(2)

**Verified validation result:** all five KPIs matched between Pandas and SQL in the completed analysis.

## 10. Power BI Dashboard Plan

### Page 1 — Executive Overview
KPI cards: Total Encounters, Total Readmissions, Readmission Rate, Average LOS, Average Medications.  
Visuals: age, gender, race, admission type, and LOS.  
Slicers: Age, Gender, Race, Admission Type, Diabetes Medication, Medication Change.

### Page 2 — Patient & Utilization Analysis
Previous inpatient, emergency, outpatient, overall prior utilization, LOS, procedures, medications, and number of diagnoses.

### Page 3 — Diagnosis & Medication Analysis
Top diagnoses by encounter volume, total readmissions, and readmission rate; diabetes medication; medication change; A1C; insulin.

### Page 4 — Interactive Readmission Explorer
Suggested hierarchy: **Age → Gender → Race → Admission Type → Diagnosis**.  
Use drill-down, drill-through, cross-filtering, tooltips, Top-N filters, conditional formatting, bookmarks, reset filters, and dynamic titles.

### Required DAX
```DAX
Total Encounters = COUNTROWS(hospital_encounters)

Total Readmissions = SUM(hospital_encounters[readmission_flag])

Readmission Rate = DIVIDE([Total Readmissions], [Total Encounters], 0)

Average LOS = AVERAGE(hospital_encounters[time_in_hospital])

Average Medications = AVERAGE(hospital_encounters[num_medications])
```

## 11. Evaluation and Interpretation

### Five Most Important Findings

1. **Overall readmission:** 3,965 of 10,000 encounters were readmitted, for an overall rate of **39.65%**.
2. **Older age groups:** The 80–90 group had the highest age-specific rate (**44.32%**). The 70–80 group was also high (**43.04%**) and large (**2,595 encounters**), making it operationally meaningful.
3. **Prior utilization:** Readmission increased substantially with prior inpatient use: **34.36%** with zero prior inpatient visits, **51.17%** with one, and **62.74%** with two.
4. **Congestive heart failure:** CHF combined high risk and meaningful volume: **636 encounters, 342 readmissions, 53.77% readmission rate**.
5. **Medication-related patterns:** Diabetes medication and medication changes were associated with higher observed readmission rates, but these differences may reflect underlying clinical complexity rather than a causal medication effect.

### Data-Quality and Interpretation Limitations
- `A1Cresult` has approximately **83.79% missingness**.
- `max_glu_serum` has approximately **93.36% missingness**.
- Very small categories can produce unstable rates such as 100%.
- The dataset is encounter-level; 10,000 encounters do not necessarily represent 10,000 unique patients.
- The analysis is descriptive and cannot establish causation.
- Additional information such as disease severity, adherence, socioeconomic factors, follow-up care, and reason for readmission would improve interpretation.

### Final Business Interpretation
The strongest actionable pattern is prior healthcare utilization. Older high-volume segments and CHF also combine above-average readmission with meaningful encounter volume. These groups may be useful for targeted discharge planning, medication reconciliation, follow-up, and care coordination, while any intervention should be evaluated with additional clinical context.

## 12. Deployment / Submission Checklist

- Jupyter Notebook: organized analysis, advanced Pandas, visualizations, SQL, validation, interpretation
- Cleaned CSV: `diabetes_clean.csv`
- SQL: at least 12 meaningful queries with CASE WHEN, CTE, window function, ranking, and conditional aggregation
- Power BI: 4-page interactive dashboard with DAX measures
- Executive Summary: major findings, interpretation, limitations, recommendations
- Publish to Power BI Service if available and verify that record-level health information is not unnecessarily exposed